# Camada Silver — Limpeza e Padronização

A camada **Silver** recebe os dados brutos da camada **Bronze** e aplica **4 transformações** para deixá-los prontos para análise:

1. **Tipagem** — Garantir que cada coluna tenha o tipo correto (ex: texto, número inteiro, número decimal, data)
2. **Remoção de Nulos** — Remover linhas que não têm valor em colunas essenciais (ex: um cliente sem ID é inútil)
3. **Deduplicação** — Remover linhas repetidas (ex: mesmo cliente cadastrado duas vezes)
4. **Datas** — Garantir que colunas de data estejam no formato `DATE` correto

> Pense na camada Bronze como os ingredientes crus que você comprou no mercado, e na Silver como os ingredientes já lavados, descascados e cortados — prontos para cozinhar.

In [0]:
%sql
-- ============================================================
-- SILVER: CUSTOMERS
-- ============================================================
-- Lemos da Bronze e aplicamos as 4 transformações.

CREATE OR REPLACE TABLE capgemini_trainingforthecase.silver.customers AS

WITH passo1_tipagem AS (
    -- 1. TIPAGEM: CAST garante que cada coluna tenha o tipo exato.
    --    STRING = texto | INT = numero inteiro | DOUBLE = numero decimal | DATE = data
    SELECT
        CAST(Customer_ID   AS STRING)  AS customer_id,
        CAST(Customer_Name AS STRING)  AS customer_name,
        CAST(Gender        AS STRING)  AS gender,
        CAST(Age           AS INT)     AS age,
        CAST(Age_Group     AS STRING)  AS age_group,
        CAST(Date_of_Birth AS DATE)    AS date_of_birth,        -- 4. DATAS: CAST para DATE
        CAST(Email         AS STRING)  AS email,
        CAST(Phone         AS STRING)  AS phone,
        CAST(City          AS STRING)  AS city,
        CAST(State         AS STRING)  AS state,
        CAST(Pincode       AS INT)     AS pincode,
        CAST(Registration_Date AS DATE) AS registration_date,    -- 4. DATAS: CAST para DATE
        CAST(Customer_Tier AS STRING)  AS customer_tier,
        CAST(Total_Orders  AS INT)     AS total_orders,
        CAST(Total_Spent   AS DOUBLE)  AS total_spent
    FROM capgemini_trainingforthecase.bronze.customers
),

passo2_sem_nulos AS (
    -- 2. REMOCAO DE NULOS: manter so linhas onde as colunas essenciais tem valor.
    --    Se customer_id ou email estao vazios (NULL), a linha nao serve.
    SELECT * FROM passo1_tipagem
    WHERE customer_id IS NOT NULL
      AND customer_name IS NOT NULL
      AND email IS NOT NULL
),

passo3_deduplicado AS (
    -- 3. DUPLICACAO: se o mesmo customer_id aparecer mais de uma vez,
    --    ficamos so com a linha mais recente (maior registration_date).
    --    ROW_NUMBER() numera as linhas: 1 = a mais recente.
    SELECT * FROM passo2_sem_nulos
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY customer_id
        ORDER BY registration_date DESC
    ) = 1
)

-- Resultado final ja com tipagem, sem nulos, sem duplicatas e com datas corretas
SELECT * FROM passo3_deduplicado;

In [0]:
%sql
-- ============================================================
-- SILVER: PRODUCTS
-- ============================================================

CREATE OR REPLACE TABLE capgemini_trainingforthecase.silver.products AS

WITH passo1_tipagem AS (
    -- 1. TIPAGEM: garantir tipos corretos para todas as colunas
    SELECT
        CAST(Product_ID        AS STRING) AS product_id,
        CAST(Product_Name      AS STRING) AS product_name,
        CAST(Category           AS STRING) AS category,
        CAST(Brand              AS STRING) AS brand,
        CAST(Original_Price     AS DOUBLE) AS original_price,
        CAST(Discount_Percent   AS INT)    AS discount_percent,
        CAST(Discount_Amount    AS DOUBLE) AS discount_amount,
        CAST(Selling_Price      AS DOUBLE) AS selling_price,
        CAST(Stock_Quantity     AS INT)    AS stock_quantity,
        CAST(Weight_kg          AS DOUBLE) AS weight_kg,
        CAST(Avg_Rating         AS DOUBLE) AS avg_rating,
        CAST(Total_Reviews      AS INT)    AS total_reviews
    FROM capgemini_trainingforthecase.bronze.products
),

passo2_sem_nulos AS (
    -- 2. REMOCAO DE NULOS: colunas essenciais nao podem estar vazias
    SELECT * FROM passo1_tipagem
    WHERE product_id IS NOT NULL
      AND product_name IS NOT NULL
      AND selling_price IS NOT NULL
),

passo3_deduplicado AS (
    -- 3. DUPLICACAO: um produto por ID
    SELECT * FROM passo2_sem_nulos
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY product_id
        ORDER BY product_name
    ) = 1
)

-- 4. DATAS: esta tabela nao tem colunas de data, entao as 3 transformacoes acima bastam
SELECT * FROM passo3_deduplicado;

In [0]:
%sql
-- ============================================================
-- SILVER: SALES
-- ============================================================
-- Esta tabela tem 250 mil linhas e colunas que PODEM ser nulas legitimamente:
--   Coupon_Code, Rating, Review_Text  nem todo pedido tem cupom ou avaliacao.
-- Por isso, so filtramos nulos nas colunas ESSENCIAIS.

CREATE OR REPLACE TABLE capgemini_trainingforthecase.silver.sales AS

WITH passo1_tipagem AS (
    -- 1. TIPAGEM: garantir tipos corretos
    SELECT
        CAST(Order_ID          AS STRING) AS order_id,
        CAST(Customer_ID       AS STRING) AS customer_id,
        CAST(Product_ID         AS STRING) AS product_id,
        CAST(Order_Date         AS DATE)   AS order_date,         -- 4. DATAS
        CAST(Order_Time         AS STRING) AS order_time,
        CAST(Delivery_Date      AS DATE)   AS delivery_date,      -- 4. DATAS
        CAST(Quantity           AS INT)    AS quantity,
        CAST(Unit_Price         AS DOUBLE) AS unit_price,
        CAST(Order_Value        AS DOUBLE) AS order_value,
        CAST(Shipping_Cost      AS DOUBLE) AS shipping_cost,
        CAST(Coupon_Code        AS STRING) AS coupon_code,        -- pode ser NULL (sem cupom)
        CAST(Coupon_Discount    AS DOUBLE) AS coupon_discount,
        CAST(Total_Amount       AS DOUBLE) AS total_amount,
        CAST(Payment_Mode       AS STRING) AS payment_mode,
        CAST(Order_Status       AS STRING) AS order_status,
        CAST(Rating             AS DOUBLE) AS rating,             -- pode ser NULL (sem avaliacao)
        CAST(Review_Text        AS STRING) AS review_text,        -- pode ser NULL (sem comentario)
        CAST(City               AS STRING) AS city,
        CAST(State              AS STRING) AS state,
        CAST(Customer_Age       AS INT)    AS customer_age,
        CAST(Customer_Age_Group AS STRING) AS customer_age_group
    FROM capgemini_trainingforthecase.bronze.sales
),

passo2_sem_nulos AS (
    -- 2. REMOCAO DE NULOS: so colunas essenciais
    --    Coupon_Code, Rating e Review_Text podem ficar NULL (e normal)
    SELECT * FROM passo1_tipagem
    WHERE order_id IS NOT NULL
      AND customer_id IS NOT NULL
      AND product_id IS NOT NULL
      AND order_date IS NOT NULL
      AND total_amount IS NOT NULL
),

passo3_deduplicado AS (
    -- 3. DUPLICACAO: um pedido por Order_ID
    SELECT * FROM passo2_sem_nulos
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY order_id
        ORDER BY order_date DESC
    ) = 1
)

-- Resultado final: tipagem correta, sem nulos nas colunas essenciais,
-- sem duplicatas, e com as datas (order_date, delivery_date) no formato DATE
SELECT * FROM passo3_deduplicado;

In [0]:
# ============================================================
# VERIFICACAO: conferir se as tabelas Silver foram criadas corretamente
# ============================================================

silver_tables = [
    ("customers", "capgemini_trainingforthecase.silver.customers"),
    ("products", "capgemini_trainingforthecase.silver.products"),
    ("sales", "capgemini_trainingforthecase.silver.sales"),
]

for name, table in silver_tables:
    df = spark.table(table)
    count = df.count()
    print(f"{name}: {count} linhas")
    print(f"  Colunas: {df.columns}")
    print()

print("Camada Silver criada com sucesso!")